In [0]:
dbutils.widgets.text("catalog", "vp_rm20040829")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

##Stations

In [0]:
stations_df = spark.read.table(f"{catalog}.bronze.stations")

#trim, uppercase, format date not null
stations_df = (
    stations_df
    .withColumn("station_id", trim(upper(col("station_id"))))
    .withColumn("station_name", trim(col("station_name")))
    .withColumn("city", trim(initcap(col("city"))))
    .withColumn("site_type", trim(upper("site_type")))
    .withColumn("opened_on", to_date(col("opened_on"), "yyyy-MM-dd"))
)

stations_df = (
    stations_df
    .drop("_ingested_at")
    .drop("_source_file")
    )

stations_df = stations_df.dropDuplicates()
stations_df = stations_df.dropna(how = "all")

stations_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stations")
stations_df.count()

In [0]:
spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.stations IS 'This table contains transformed stations';
""")

##Tariffs

In [0]:
tariffs_df = spark.read.table(f"{catalog}.bronze.tariffs")

#trim, uppercase, decimal(10,2)
tariffs_df = (
    tariffs_df
    .withColumn("connector_type", trim(upper(col("connector_type"))))
    .withColumn("time_band", trim(upper(col("time_band"))))
    .withColumn("rate_per_kwh", col("rate_per_kwh").cast("decimal(10,2)"))
    .withColumn("idle_fee_per_min", col("idle_fee_per_min").cast("decimal(10,2)"))
    .withColumn("free_idle_minutes", col("free_idle_minutes").cast("int"))
)

tariffs_df = (
    tariffs_df
    .drop("_ingested_at")
    .drop("_source_file")
    )

tariffs_df = tariffs_df.dropDuplicates()
tariffs_df = tariffs_df.dropna(how = "all")

tariffs_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.tariffs")
tariffs_df.count()

In [0]:
spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.tariffs IS 'This table contains transformed tariffs';
""")

##Chargers

In [0]:
chargers_df = spark.read.table(f"{catalog}.bronze.chargers")

chargers_df = (
    chargers_df
    .withColumn("station_id", trim(upper(col("station_id"))))
    .withColumn("charger_id", trim(upper(col("charger_id"))))
    .withColumn("connector_type", trim(upper(col("connector.type"))))
    .withColumn("is_active", col("is_active").cast("boolean"))
    .withColumn("snapshot_date", to_date(col("snapshot_date"), "yyyy-MM-dd"))
    .withColumn("commissioned_on", to_date(col("commissioned_on"), "yyyy-MM-dd"))
    .withColumn("max_kw", col("connector.max_kw").cast("decimal(10,2)"))
)

chargers_df = (
    chargers_df
    .drop("_ingested_at")
    .drop("_source_file")
    )

chargers_df = chargers_df.dropDuplicates()
chargers_df = chargers_df.dropna(how = "all")

chargers_df.createOrReplaceTempView("chargers_df")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.chargers
AS
SELECT *, snapshot_date AS last_changed_snapshot_date
FROM chargers_df
""")

#enable change data feed
spark.sql(f"ALTER TABLE {catalog}.silver.chargers SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

#merge into for scd type 1
spark.sql(f"""
MERGE INTO {catalog}.silver.chargers AS t
USING (
  SELECT *, snapshot_date AS last_changed_snapshot_date
  FROM (
    SELECT *, row_number() OVER (PARTITION BY station_id, charger_id ORDER BY snapshot_date DESC) AS rn
    FROM chargers_df
  ) WHERE rn = 1
) AS s
ON t.station_id = s.station_id AND t.charger_id = s.charger_id
WHEN MATCHED AND (
  t.connector_type IS DISTINCT FROM s.connector_type OR
  t.is_active IS DISTINCT FROM s.is_active OR
  t.commissioned_on IS DISTINCT FROM s.commissioned_on OR
  t.max_kw IS DISTINCT FROM s.max_kw
)
THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.chargers IS 'This table contains transformed chargers';
""")

In [0]:
%sql
select count(*) from vp_rm20040829.silver.chargers

##Members

In [0]:
members_df = spark.read.table(f"{catalog}.bronze.members")


members_df = (
    members_df
    .withColumn("member_id", trim(upper(col("member_id"))))
    .withColumn("full_name", trim(col("full_name")))
    .withColumn("email", trim(lower(col("email"))))
    .withColumn("phone", regexp_replace(col("phone"), "[^0-9]", ""))
    .withColumn("phone", substring(col("phone"), -10, 10))
    .withColumn("phone", when(length(col("phone")) < 10, lit(None)).otherwise(col("phone")))
    .withColumn("city", trim(initcap(col("city"))))
    .withColumn("plan", trim(upper(col("plan"))))
    .withColumn("joined_on", try_to_date(col("joined_on"), "yyyy-MM-dd"))
    .withColumn("joined_on", when(length(col("joined_on")) == 10, try_to_date(col("joined_on"), "dd/MM/yyyy")).otherwise(col("joined_on")))
    .withColumn("change_effective_date", col("_ingested_at").cast("date"))
)

members_df = (
    members_df
    .drop("_ingested_at")
    .drop("_source_file")
    )

members_df = members_df.dropDuplicates()
members_df = members_df.dropna(how = "all")

members_df.createOrReplaceTempView("members_df")


members_latest = spark.sql("""
SELECT * FROM (
  SELECT *, row_number() OVER (PARTITION BY member_id ORDER BY change_effective_date DESC) AS rn
  FROM members_df
) WHERE rn = 1
""")
members_latest.createOrReplaceTempView("members_latest")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.members (
  member_id STRING,
  full_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  plan STRING,
  joined_on DATE,
  effective_from DATE,
  effective_to DATE,
  is_current BOOLEAN
)
""")

spark.sql(f"""
MERGE INTO {catalog}.silver.members AS t
USING members_latest AS s
ON t.member_id = s.member_id AND t.is_current = true
WHEN MATCHED AND (
  (t.full_name IS DISTINCT FROM s.full_name
   OR t.email IS DISTINCT FROM s.email
   OR t.phone IS DISTINCT FROM s.phone)
  AND t.plan IS NOT DISTINCT FROM s.plan
  AND t.city IS NOT DISTINCT FROM s.city
)
THEN UPDATE SET t.full_name = s.full_name, t.email = s.email, t.phone = s.phone
""")

spark.sql(f"""
MERGE INTO {catalog}.silver.members AS t
USING members_latest AS s
ON t.member_id = s.member_id AND t.is_current = true
WHEN MATCHED AND (t.plan IS DISTINCT FROM s.plan OR t.city IS DISTINCT FROM s.city)
THEN UPDATE SET t.effective_to = s.change_effective_date, t.is_current = false
""")

spark.sql(f"""
INSERT INTO {catalog}.silver.members
SELECT
  s.member_id, s.full_name, s.email, s.phone, s.city, s.plan, s.joined_on,
  s.change_effective_date AS effective_from, NULL AS effective_to, true AS is_current
FROM members_latest AS s
LEFT JOIN {catalog}.silver.members AS t
  ON s.member_id = t.member_id AND t.is_current = true
WHERE t.member_id IS NULL
""")

In [0]:
%sql
select * from vp_rm20040829.silver.members

In [0]:
spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.members IS 'This table contains transformed members';
""")